# Navojoa climate — Data cleaning and EDA
Modernized companion to the original BD_Climatologica_Navojoa notebook. Original analysis is preserved in the repository root.
Data source: SMN station 26131, recorded 1931–1993. Available observations and month/year coverage are unequal.


In [1]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import seaborn as sns
root = Path.cwd()
if not (root / 'ClimatologiaNavojoa.txt').exists(): root = root.parent
sys.path.insert(0, str(root / 'scripts'))
from climate_processing import load_daily, monthly_climatology, annual_summary
df = load_daily()
df.head()


,date,precip_mm,tmax_c,tmin_c,year,month
0,1931-01-01,0.0,28.0,8.0,1931,1
1,1931-01-02,0.0,29.0,7.0,1931,1
2,1931-01-03,0.0,29.0,9.0,1931,1
3,1931-01-04,0.0,29.0,6.0,1931,1
4,1931-01-05,0.0,26.0,5.0,1931,1


## Audit: observations and missingness


In [2]:
print(df[['date','precip_mm','tmax_c','tmin_c']].isna().sum())
print('Rows:',len(df),'date range:',df.date.min(),df.date.max())
df[['precip_mm','tmax_c','tmin_c']].describe()


date           0
precip_mm     34
tmax_c        82
tmin_c       111
dtype: int64
Rows: 13404 date range: 1931-01-01 00:00:00 1993-07-31 00:00:00


,precip_mm,tmax_c,tmin_c
count,13370.000000,13322.000000,13293.000000
mean,1.015999,34.195016,14.528549
std,6.192861,6.081358,7.343251
min,0.000000,11.000000,-7.000000
25%,0.000000,30.000000,9.000000
50%,0.000000,35.000000,13.000000
75%,0.000000,39.000000,21.000000
max,254.000000,52.000000,40.000000


## Seasonal daily means
Daily mean precipitation (mm per observed day) is not the same as the total monthly precipitation.


In [3]:
climate = monthly_climatology(df)
display(climate)
fig,axes=plt.subplots(1,2,figsize=(13,4))
sns.barplot(data=climate,x='month',y='mean_daily_precip_mm',ax=axes[0],color='#2878a6')
axes[0].set(ylabel='Mean daily precipitation (mm)',xlabel='Month')
axes[1].plot(climate.month,climate.mean_tmax_c,label='Mean Tmax')
axes[1].plot(climate.month,climate.mean_tmin_c,label='Mean Tmin')
axes[1].set(xlabel='Month',ylabel='Temperature (°C)')
axes[1].legend()
plt.tight_layout()
plt.show()


,month,mean_daily_precip_mm,precipitation_observations,mean_tmax_c,mean_tmin_c
0,1,0.606265,1213,27.084158,6.979373
1,2,0.443551,1132,28.838938,7.473451
2,3,0.215277,1211,31.239274,8.545301
3,4,0.011008,1181,34.236441,10.594915
4,5,0.000925,1081,36.686486,13.870270
5,6,0.383523,1141,38.966696,20.997371
6,7,2.692464,1181,38.682485,23.886109
7,8,3.696030,1083,38.185325,23.190969
8,9,2.340299,1072,38.190476,22.279048
9,10,0.809315,1095,36.548624,17.554128


## Distribution of observed daily temperatures


In [4]:
fig,axes=plt.subplots(1,2,figsize=(12,4))
sns.histplot(df.tmax_c.dropna(),bins=30,kde=True,ax=axes[0])
sns.histplot(df.tmin_c.dropna(),bins=30,kde=True,ax=axes[1])
axes[0].set(xlabel='Daily Tmax (°C)')
axes[1].set(xlabel='Daily Tmin (°C)')
plt.tight_layout()
plt.show()


## Caveats
Data completeness differs by month and year. Do not interpret uncorrected annual means and totals as a climate trend. Original notebook includes additional boxplots and distributions.
